In [1]:
# !pip install pytorch_lightning

In [2]:
import os, random, math, pickle
import pandas as pd
import numpy as np
from datetime import datetime

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import pytorch_lightning as pl
from torch.utils.data import random_split
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping

# Set environment variables for reproducibility and safety
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
import warnings
warnings.filterwarnings('ignore')
from sklearn.metrics import precision_score, recall_score, accuracy_score

# # 1. Configuration & Seeding
# SEED = 42
# random.seed(SEED)
# np.random.seed(SEED)
# torch.manual_seed(SEED)
# if torch.cuda.is_available():
#     torch.cuda.manual_seed_all(SEED)

c:\Users\vtruong1\AppData\Local\Programs\Python\Python311\Lib\site-packages\torchmetrics\__init__.py:31: UserWarning: A NumPy version >=1.23.5 and <2.3.0 is required for this version of SciPy (detected version 2.3.4)
  import scipy.signal


In [3]:
# name = 'book'
# N_STATIC_RELATION = 20
# N_CLUSTER = 15

# USER_START_ID = 1
# USER_END_ID = 2890
# ITEM_START_ID = 2891  # book: 2891
# ITEM_END_ID = 11584
# STATIC_ENTITY_START_ID = 11585  # book: 2891
# STATIC_ENTITY_END_ID = 38885

# N_ENTITY = STATIC_ENTITY_END_ID
# N_RELATION = N_STATIC_RELATION + N_CLUSTER

In [4]:
name = 'movie'
N_STATIC_RELATION = 20
N_CLUSTER = 15

USER_START_ID = 1
USER_END_ID = 1439
ITEM_START_ID = 1440  # book: 2891
ITEM_END_ID = 6335
STATIC_ENTITY_START_ID = 6336  # book: 2891
STATIC_ENTITY_END_ID = 54535

N_ENTITY = STATIC_ENTITY_END_ID
N_RELATION = N_STATIC_RELATION + N_CLUSTER

#### 1.1 KGCN Model

In [5]:
from collections import defaultdict

class KnownledgeGraph():
    def __init__(self, graph_df, nbr_sample_size=1):
        self.graph_df = graph_df
        self.graph_nbr_sample_size = nbr_sample_size
        self.graph_entities = None
        self.graph_relations = None
        self.graph = defaultdict(set)

    def build(self):
        df = self.graph_df

        # Build the knowledge graph with sets
        for head_id, relation_id, tail_id, _, _, _ in df.values:
            self.graph[head_id].add((tail_id, relation_id))
            self.graph[tail_id].add((head_id, relation_id))

    def get_neighbors(self, entity_id, sample_size=8):
        """Get neighbors for an entity with optional sampling"""
        if sample_size is None:
            sample_size = self.graph_nbr_sample_size

        entity_id = entity_id.item() if torch.is_tensor(entity_id) else entity_id
        neighbors = list(self.graph.get(entity_id, set()))
        n_neighbors = len(neighbors)

        if n_neighbors == 0:
            return torch.tensor([], dtype=torch.long), torch.tensor([], dtype=torch.long)

        if n_neighbors >= sample_size:
            sampled_indices = np.random.choice(n_neighbors, size=sample_size, replace=False)
        else:
            sampled_indices = np.random.choice(n_neighbors, size=sample_size, replace=True)

        sampled_neighbors = [neighbors[i] for i in sampled_indices]
        adj_entities = torch.tensor([n for n, _ in sampled_neighbors], dtype=torch.long)
        adj_relations = torch.tensor([r for _, r in sampled_neighbors], dtype=torch.long)

        return adj_entities, adj_relations

In [6]:
class SumAggregator(nn.Module):
    def __init__(self, embedding_dim):
        super(SumAggregator, self).__init__()
        self.linear = nn.Linear(embedding_dim, embedding_dim)

    def forward(self, neighbor_embs, central_embs):
        """
        neighbor_embs: Tensor of shape (batch, emb_dim)
        central_embs: Tensor of shape (batch, emb_dim)
        """

        # Combine with central entity embedding
        combined = neighbor_embs + central_embs  # shape: (batch, emb_dim) ([1204, 64])

        # Linear + activation
        output = torch.tanh(self.linear(combined))  # shape: (batch, emb_dim) torch.Size([1204, 64])
        return output

In [7]:
class KGCN(pl.LightningModule):
    def __init__(self, graph: KnownledgeGraph, item_pool, item2pool_idx, num_neg_samples, embedding_dim, lr, loss_margin, temperature ):
        super().__init__()
        self.save_hyperparameters()

        model_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model_device = model_device

        self.graph = graph
        self.item_pool = item_pool # Move item_pool to the correct device .to(model_device)
        self.item2pool_idx = item2pool_idx
        

        self.num_neg_samples = num_neg_samples

        self.embedding_dim = embedding_dim

        # Embedding layersnum_neg_samples
        self.entity_embedding = nn.Embedding(N_ENTITY + 1, embedding_dim, padding_idx = 0).to(model_device)
        self.relation_embedding = nn.Embedding(N_RELATION + 1, embedding_dim, padding_idx = 0).to(model_device)

        # Aggregator
        self.aggregator = SumAggregator(embedding_dim).to(model_device)

        self.dropout = nn.Dropout(p=0.1)
        self.loss_function = torch.nn.MarginRankingLoss(margin= loss_margin)
        self.temperature= temperature

    def setup(self, stage=None):
        self.train_user_pos_items = self.trainer.datamodule.train_user_pos_items
        self.val_user_pos_items = self.trainer.datamodule.val_user_pos_items

    @staticmethod
    def hit_at_k(pred_items, true_items, k):
        hits = 0
        for pred, true in zip(pred_items, true_items):
            # Count if any true item is in top-k predictions
            if len(set(pred[:k]) & set(true)) > 0:
                hits += 1
        return hits / len(true_items)

    @staticmethod
    def ndcg_at_k(pred_items, true_items, k):
        ndcg = 0.0
        for pred, true in zip(pred_items, true_items):
            gains = []
            for idx, item in enumerate(pred[:k]):
                gains.append(1 if item in true else 0)
            ideal_gains = [1] * min(len(true), k)
            dcg = sum(g / math.log2(i+2) for i, g in enumerate(gains))
            idcg = sum(g / math.log2(i+2) for i, g in enumerate(ideal_gains))
            ndcg += dcg / idcg if idcg > 0 else 0
        return ndcg / len(true_items)

    @staticmethod
    def recall_at_k(pred_items, true_items, k):
        recall = 0.0
        for pred, true in zip(pred_items, true_items):
            recall += len(set(pred[:k]) & set(true)) / len(true)
        return recall / len(true_items)

    @staticmethod
    def precision_at_k(pred_items, true_items, k):
        precision = 0.0
        for pred, true in zip(pred_items, true_items):
            precision += len(set(pred[:k]) & set(true)) / k
        return precision / len(true_items)

    def enhance_item_embs(self, user_ids, item_ids):
        '''
        user_ids: (batch)
        item_ids: (batch). Mỗi phần tử trong batch chỉ 1 item.
        '''

        neighbor_ids, relation_ids = self.graph.get_neighbors(item_ids) # (batch, N). Mỗi phần tử trong batch có nhiều hơn 1 neighbor

        user_embs = self.entity_embedding(user_ids) # (batch, embed_dim)
        item_embs = self.entity_embedding(item_ids) # (batch, embed_dim) 

        neighbor_embs = self.entity_embedding(neighbor_ids) # (batch, N, embed_dim)
        relation_embs = self.relation_embedding(relation_ids) # (batch, N, embed_dim)

        ################ User - Relation Attention ################
        # Expand user_embs to match neighbor dimension
        user_embs_norm = F.normalize(user_embs, p=2, dim=1)
        relation_embs_norm = F.normalize(relation_embs, p=2, dim=2)
        scores = (user_embs_norm.unsqueeze(1) * relation_embs_norm).sum(dim=2)

        attention_weights = F.softmax(scores, dim=1).unsqueeze(-1)  # [1204, 8, 1]
        weighted_neighbor_embs = neighbor_embs * attention_weights  # [1204, 8, 64]
        weighted_neighbor_embs = weighted_neighbor_embs.mean(dim=1) # [1204, 64]
        ################ User - Relation Attention ################

        updated_item_embs  = self.aggregator(weighted_neighbor_embs, item_embs) #([1204, 64])

        return user_embs, updated_item_embs

    def enhance_multi_item_embs(self, user_ids, multi_item_ids):
        '''
        user_ids: (batch)
        multi_item_ids: (batch, N). Mỗi phần tử trong batch có nhiều hơn 1 item.
        '''

        user_embs = self.entity_embedding(user_ids) # (batch, embed_dim)
        multi_item_embs = self.entity_embedding(multi_item_ids) # (batch, N, embed_dim)   

        # Giả sử item_embs có shape [batch, N, dim]
        batch_size, N, dim = multi_item_embs.shape

        # 1. Nhân bản user_embs N lần để mỗi negative item đều được ghép cặp với user đó
        # Shape: [batch, dim] -> [batch, 1, dim] -> [batch, N, dim]
        user_embs_expanded = user_embs.unsqueeze(1).expand(-1, N, -1)

        # 2. Đập bẹp (Flatten) cả 2 tensor về dạng 2D 
        # Shape mới: [batch * N, dim]
        flat_user_embs = user_embs_expanded.reshape(batch_size * N, dim)
        flat_item_embs = multi_item_embs.reshape(batch_size * N, dim)

        # 3. Đưa qua hàm update() nguyên bản của bạn
        # Hàm vẫn nghĩ nó đang xử lý một batch lớn có kích thước là (batch * N)
        flat_updated_embs = self.enhance_item_embs(flat_user_embs, flat_item_embs)

        # 4. Phục hồi lại shape 3D ban đầu
        # Shape: [batch * N, dim] -> [batch, N, dim]
        updated_multi_item_embs = flat_updated_embs.reshape(batch_size, N, dim)

        return user_embs, updated_multi_item_embs

    def forward(self, batch):
        '''
        user_ids: (batch)
        item_ids: (batch). Mỗi phần tử trong batch chỉ 1 item.
        '''
        user_ids, item_ids = batch

        user_embs, updated_item_embs = self.enhance_item_embs(user_ids, item_ids)

        return user_embs, updated_item_embs

    def compute_loss(self, batch, user_embs, updated_item_embs):
        user_ids, item_ids = batch

        pos_item_embs = updated_item_embs

        user_embs_norm = F.normalize(user_embs, p=2, dim=1)
        pos_item_embs_norm = F.normalize(pos_item_embs, p=2, dim=1)
        pos_scores = (user_embs_norm * pos_item_embs_norm).sum(dim=1)

        ########################## Random sample (batch, self.num_sample_items, embedd_dim) ##################
        users = user_ids.cpu().tolist()
        pool_size = len(self.item_pool)
        neg_item_ids_list = []

        # 2. Lấy mẫu âm có kiểm tra loại trừ cho từng user
        for u in users:
            # ÉP SANG SET NGAY TẠI ĐÂY: Chuyển list thành set để phép kiểm tra 'not in' chạy với tốc độ O(1)
            pos_items_of_u = set(self.train_user_pos_items[u])

            user_negs = []
            # Tiếp tục bốc ngẫu nhiên cho đến khi đủ số lượng num_neg_samples
            while len(user_negs) < self.num_neg_samples:
                rand_idx = random.randint(0, pool_size - 1)

                # Nếu item_pool là Tensor, nhớ dùng .item() để lấy số nguyên thuần Python
                sampled_item = self.item_pool[rand_idx].item()

                # CHỐT CHẶN SIÊU TỐC: Chỉ chấp nhận nếu item vừa bốc KHÔNG nằm trong set
                if sampled_item not in pos_items_of_u:
                    user_negs.append(sampled_item)

            neg_item_ids_list.append(user_negs)

        # 3. Chuyển đổi list kết quả (batch_size, num_neg_samples) thành Tensor và đẩy thẳng lên GPU
        neg_item_ids = torch.tensor(neg_item_ids_list, dtype=torch.long, device=self.model_device) # torch.Size([512, n_sample 50])

        # 4. Lấy embeddings (Lưu ý: Không dùng F.normalize ở đây như chúng ta đã thống nhất)
        neg_item_embs = self.entity_embedding(neg_item_ids) # torch.Size([512, n sample 50, dim 64])

        # 5. Tính điểm Dot Product cho toàn bộ mẫu âm (Shape: [batch_size, num_neg_samples])
        neg_item_embs_norm = F.normalize(neg_item_embs, p=2, dim=2)
        neg_scores = (user_embs_norm.unsqueeze(1) * neg_item_embs_norm).sum(dim=2) # torch.Size([512, n_sample 50])

        # 6. Chọn số lượng hard negatives (Ví dụ: từ 20 mẫu âm bốc ra, chọn 5 mẫu khó nhất)
        k = 10

        # Lấy trực tiếp VALUE (điểm số cao nhất) thay vì INDICES
        # Shape của hard_neg_scores: [batch_size, k]
        hard_neg_scores = torch.topk(neg_scores, k=k, dim=1).values # # torch.Size([512, 10)

        # 7. Tính trung bình điểm của các hard negatives này
        # Shape cuối cùng: [batch_size] -> Sẵn sàng đưa vào hàm Loss
        neg_scores = hard_neg_scores.mean(dim=1) #torch.Size([512])
        ########################## Random sample (batch, self.num_sample_items, embedd_dim) ##################

        ####################### Compute Loss #######################
        scores = torch.cat([pos_scores, neg_scores], dim=0)
        labels = torch.cat([torch.ones_like(pos_scores), torch.zeros_like(neg_scores)], dim=0)

        # Use binary_cross_entropy_with_logits instead of binary_cross_entropy
        # because scores are raw logits (dot products), not probabilities (0-1)
        loss = F.binary_cross_entropy_with_logits(scores, labels)

        # loss = - F.logsigmoid(pos_scores - neg_scores).mean() #### chay duoc

        # margin = 0.2
        # loss = F.relu(neg_scores - pos_scores + margin).mean()

        # loss = self.loss_function(pos_scores, neg_scores, target=torch.ones_like(pos_scores))
        ####################### Compute Loss #######################

        return loss, pos_scores, neg_scores

    def training_step(self, batch, batch_idx):
        user_embs, updated_item_embs = self(batch)
        train_loss, ps, ns = self.compute_loss(batch, user_embs, updated_item_embs)

        self.log("train_loss", train_loss, on_epoch=True, prog_bar=True)

        return train_loss

    def validation_step(self, batch, batch_idx):
        user_ids, item_ids = batch

        user_embs, updated_item_embs = self(batch)

        val_loss, ps, ns = self.compute_loss(batch, user_embs, updated_item_embs)
        self.log('val_loss', val_loss, prog_bar=True, logger=True)

        ################ Calculate metrics
        pool_ids_tensor = torch.tensor(self.item_pool, dtype=torch.long, device=self.device)
        # pool_item_embs = self.entity_embedding(pool_ids_tensor) #shape torch.Size([1023, 64])
        pool_item_embs = self.enhance_multi_item_embs(user_ids, self.item_pool)

        # # Tính scores (Shape: [batch_size, num_items_in_pool])
        user_embs_norm = F.normalize(user_embs, p=2, dim=1) # torch.Size([512, 64])
        pool_item_embs_norm = F.normalize(pool_item_embs, p=2, dim=1) # torch.Size([N = 1023, 64])
        scores = (user_embs_norm.unsqueeze(1) * pool_item_embs_norm).sum(dim=2) # torch.Size([512, 1023])

        # ==========================================
        # MASKING SIÊU TỐC VỚI ADVANCED INDEXING
        # ==========================================
        # 1. Gom tất cả các tọa độ cần Mask vào 2 list
        # for i, u in enumerate(user_ids.tolist()):
        #     trained_items = self.train_user_pos_items[u]
        #     cols = []
        #     for item in trained_items:
        #         item_id = self.item2pool_idx[item]
        #         cols.append(item_id) # Cột tương ứng của item đó
        #         scores[i, cols] = float('-inf')
        
        k_values = [10]  # Example: you can add more values as needed

        for k in k_values:
            topk_relative_indices = torch.topk(scores, k=k, dim=1).indices

            # 4. Map ngược ra ID thực tế bằng cách truyền thẳng vào pool_ids_tensor
            # Lưu ý: pool_ids_tensor chính là torch.tensor(self.item_pool) mà bạn đã tạo ở trên
            # Shape: [batch_size, k]
            topk_actual_ids = pool_ids_tensor[topk_relative_indices]

            # 5. (Tùy chọn) Ép về Python list dạng [ [id1, id2...], [id3, id4...] ]
            # Để đưa vào các hàm tính Hit@10 hoặc NDCG@10
            topk_items = topk_actual_ids.tolist()


            true_items = []  # (1024, variable length), each user may have multiple positive items
            for u in user_ids.tolist():
                adjusted_val_items = [item for item in self.val_user_pos_items[u]]
                true_items.append(adjusted_val_items)


            # Compute metrics for this k
            hit = self.hit_at_k(topk_items, true_items, k)
            ndcg = self.ndcg_at_k(topk_items, true_items, k)
            recall = self.recall_at_k(topk_items, true_items, k)
            precision = self.precision_at_k(topk_items, true_items, k)

            # Log metrics dynamically
            self.log(f"val_hit@{k:02d}", hit, prog_bar=True)
            self.log(f"val_recall@{k:02d}", recall, prog_bar=True)
            self.log(f"val_precision@{k:02d}", precision, prog_bar=True)
            self.log(f"val_ndcg@{k:02d}", ndcg, prog_bar=True)

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.hparams.lr, weight_decay=5e-4)
        scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.75)
        return [optimizer], [scheduler]

#### 1.2 DataModule

In [ ]:
class RecommenderDataModule(pl.LightningDataModule):
    def __init__(self, interaction_df, graph: KnownledgeGraph, batch_size, val_size):
        super().__init__()
        self.interaction_df = interaction_df
        self.batch_size = batch_size
        self.val_size = val_size
        self.graph = graph

    def prepare_data(self):
        # Load interactions
        df_inter = self.interaction_df

        # Build positive interactions
        dataset = []    # user_id, entity_id (item_id), neighbor_ids, relation_ids,
        for _, row in df_inter.iterrows():
            u = row['user_id']
            e = row['entity_id']
            dataset.append((u, e))

        # Split interactions for validation
        train_size = int(len(dataset) * (1 - self.val_size))
        val_size = len(dataset) - train_size
        self.train_dataset, self.val_dataset = random_split(dataset, [train_size, val_size])

        train_user_pos_items = defaultdict(set)
        for u, i in self.train_dataset:
            train_user_pos_items[u].add(i)

        val_user_pos_items = defaultdict(set)
        for u, i, in self.val_dataset:
            val_user_pos_items[u].add(i)

        self.train_user_pos_items = train_user_pos_items
        self.val_user_pos_items = val_user_pos_items

    def train_dataloader(self):
        return DataLoader(self.train_dataset, batch_size=self.batch_size, shuffle=True)

    def val_dataloader(self):
        return DataLoader(self.val_dataset, batch_size=self.batch_size, shuffle=False)

#### 1.4 Main

In [9]:
tckg_df = pd.read_csv(f'./data/{name}_TCKG.csv')
graph_df = tckg_df[tckg_df['relation_id'] <= 20]

interaction_df = tckg_df[tckg_df['relation_id'] >=21]

interaction_df = interaction_df.rename(columns={'head_id': 'user_id', 'tail_id': 'entity_id'})
unique_items = interaction_df['entity_id'].unique()
item_pool = torch.tensor(unique_items, dtype=torch.long)
item2pool_idx = {item_id: idx for idx, item_id in enumerate(unique_items)}

graph = KnownledgeGraph(graph_df)
graph.build()

data_module = RecommenderDataModule(interaction_df, graph, batch_size=512, val_size=0.2)
data_module.prepare_data()


ValueError: not enough values to unpack (expected 4, got 2)

In [ ]:
from pytorch_lightning import Trainer

model = KGCN(
        graph = graph,
        item_pool = item_pool,
        item2pool_idx= item2pool_idx,
        num_neg_samples = 50,
        embedding_dim= 64,
        lr= 0.01,
        loss_margin = 0.5,
        temperature = 1
    )

early_stop_callback = EarlyStopping(
    monitor="val_loss",     # metric to monitor
    patience=50,            # number of epochs with no improvement after which training will be stopped
    mode="min",             # 'min' because we want to minimize val_loss
    verbose=True
)

checkpoint_hit = ModelCheckpoint(
        monitor="val_loss",
        dirpath="./checkpoints",
        filename=f"{name}-{{epoch:02d}}-{{val_loss:.3f}}{{val_hit@10:.3f}}",
        save_top_k=1,
        mode="max",
)

trainer = Trainer(
        num_sanity_val_steps=0,
        max_epochs=20,
        accelerator="auto",
        callbacks=[checkpoint_hit, early_stop_callback],
    )

trainer.fit(model, data_module)

In [21]:
path = r"C:\Users\vtruong1\OneDrive - Intel Corporation\Desktop\Van Hao\Study\21. Luan Van\03. Learn Embedding\checkpoints\movie-epoch=21-val_loss=0.322val_hit@10=0.120.ckpt"
model = KGCN.load_from_checkpoint(path)

In [23]:
user_id = 8
user_emb = model.entity_embedding(torch.tensor(user_id))
# print(user_emb)

item_id = 5347
item_emb = model.entity_embedding(torch.tensor(item_id))
# print(item_emb)

adj_entities, adj_relations = graph.get_neighbors(item_id)
print(f"adj_entities: {adj_entities}")
print(f"adj_relations: {adj_relations}")

adj_entities: tensor([14990, 19930, 17078, 10805, 10203, 34841, 10174, 11698])
adj_relations: tensor([20, 17, 20, 20, 20, 20, 20, 20])
